# Pipeline evaluation
To execute the enviroments just execute the following cells.

In [12]:
%load_ext autoreload
%autoreload 2

from model_card_generation_pipeline import ModelCardGenerator
from utils.evaluation_tools import calculate_exact_match_f1, calculate_bertscore_complete
from tqdm.notebook import tqdm

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


[autoreload of utils.evaluation_tools failed: Traceback (most recent call last):
  File "/home/jovyan/.local/lib/python3.11/site-packages/IPython/extensions/autoreload.py", line 273, in check
    superreload(m, reload, self.old_objects)
  File "/home/jovyan/.local/lib/python3.11/site-packages/IPython/extensions/autoreload.py", line 471, in superreload
    module = reload(module)
             ^^^^^^^^^^^^^^
  File "/opt/conda/envs/benchmarkingml4kge/lib/python3.11/importlib/__init__.py", line 169, in reload
    _bootstrap._exec(spec, module)
  File "<frozen importlib._bootstrap>", line 621, in _exec
  File "<frozen importlib._bootstrap_external>", line 936, in exec_module
  File "<frozen importlib._bootstrap_external>", line 1074, in get_code
  File "<frozen importlib._bootstrap_external>", line 1004, in source_to_code
  File "<frozen importlib._bootstrap>", line 241, in _call_with_frames_removed
  File "/home/jovyan/BenchmarkingML4KGEmodelcards/BenchmarkingML4KGE_extraction/final_pipel

In [ ]:
import os 
from utils.XMLParser import XMLParser
import json
from pathlib import Path

with open("data/pwc_final.json", 'r', encoding='utf-8') as f:
    pwc_json = json.load(f)

required_fields = ['paper_repo', 'tasks', 'model_category', 'model_name']
            
reduced_dataset = [
    paper for paper in pwc_json 
    if all(paper.get(field) for field in ["Datasets", "Metrics", "model_name"])
]
#Lo he cambiado para que coja al menos 172 ejemplos, porque con 26 es demasiado poco

print(f"{len(reduced_dataset)} papers found with values in the required fields.")

clean_dataset = []

for paper in reduced_dataset:
    xml_path = paper.get("local_xml_path")
    if not xml_path:
        print(f"No XML found for {paper.get('title')}")
        continue
    
    archive=Path(xml_path)
    filename=archive.name
    filename=filename.replace("\\","/")

    base_path = Path(os.getcwd())
    xml_path = base_path / "data" / filename

    parser=XMLParser(xml_path)
    full_text = parser.get_full_text()
    abstract = parser.get_abstract()
    section_dict = parser.get_sections(target_sections=['Experiments','Evaluation','Results'])
    sections = "\n\n".join(section_dict.values())

    lightocr_json_file = filename.replace(".tei.xml","_lightonocr.json")
    lightocr_json_file = lightocr_json_file.replace("xml_outputs","lightocr_jsons")
    lightocr_json_path = base_path / "data" / lightocr_json_file

    if not lightocr_json_path.is_file():
        print(f"Tables from LightOnOCR not found for: {paper.get('title')}. Skipping paper...")
        continue
    
    with open(lightocr_json_path, 'r', encoding='utf-8') as f:
        tables = json.load(f)

    
    
    clean_paper = {
        "model_name_gt": paper.get("model_name"),
        "tasks_gt": paper.get("tasks"), 
        "category_gt": paper.get("model_category"), 
        "implementation_gt": paper.get("paper_repo"),
        "datasets_gt": paper.get("Datasets"),
        "metrics_gt": paper.get("Metrics"),
        "title": paper.get("title"),
        "local_xml_path": xml_path, 
        "pwc_abstract": paper.get("abstract"),
        "abstract": abstract,
        "full_text": full_text,
        "sections": sections,
        "tables": tables,
    }
    clean_dataset.append(clean_paper)
print(f"{len(clean_dataset)} papers left after cleaning.")
mcg = ModelCardGenerator()

157 papers found with values in the required fields.
Tables from LightOnOCR not found for: Translating Embeddings for Modeling Multi-relational Data. Skipping paper...
156 papers left after cleaning.
Initializing GLiNER...


/home/jovyan/.local/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

/home/jovyan/.local/lib/python3.11/site-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


GLiNER ready for extraction in cuda.
Initializing Qwen/Qwen3-1.7B...


...

Qwen/Qwen3-1.7B ready for extraction.
Initializing llama3.1:8b...
llama3.1:8b ready for extraction.
Initializing GlinerDatasetExtractor (fastino/gliner2-base-v1)...


[W615 11:20:53.117060560 init.cpp:767] Warning: nvfuser is no longer supported in torch script, use _jit_set_nvfuser_enabled is deprecated and a no-op (function operator())


🧠 Model Configuration
Encoder model      : microsoft/deberta-v3-base
Counting layer     : count_lstm_v2
Token pooling      : first
GlinerDatasetExtractor ready.
Initializing GlinerMetricExtractor (fastino/gliner2-base-v1)...


[W615 11:20:56.994696574 init.cpp:767] Warning: nvfuser is no longer supported in torch script, use _jit_set_nvfuser_enabled is deprecated and a no-op (function operator())


🧠 Model Configuration
Encoder model      : microsoft/deberta-v3-base
Counting layer     : count_lstm_v2
Token pooling      : first
GlinerMetricExtractor ready.
Initializing QwenMetricExtractor (qwen2.5)...
Initializing LlamaDatasetExtractor (llama3.1:8b)...
Initializing QwenMetricExtractor (qwen2.5)...
Generator ready!


In [14]:
pre_model_f1 = []
pre_tasks_f1 = []
pre_category_f1 = []
pre_implementation_f1 = []

pre_datasets_f1 = []
pre_metrics_f1 = []

pre_processing_times = []



for paper in tqdm(clean_dataset, desc="Processing papers with precise pipeline..."):
    predictions, processing_time = mcg.evaluate_pipeline("precise", paper)
    pre_processing_times.append(processing_time)
    
    if paper.get("model_name_gt"):
        pre_model_f1.append(calculate_bertscore_complete(predictions.get("model_name_prediction"), paper.get("model_name_gt"))[2])
    if paper.get("tasks_gt"):
        pre_tasks_f1.append(calculate_bertscore_complete(predictions.get("tasks_prediction"), paper.get("tasks_gt"))[2])
    if paper.get("category_gt"):
       pre_category_f1.append(calculate_bertscore_complete(predictions.get("category_prediction"), paper.get("category_gt"))[2])
    if paper.get("implementation_gt"):
        pre_implementation_f1.append(calculate_bertscore_complete(predictions.get("implementation_prediction"), paper.get("implementation_gt"))[2])
    
    if paper.get("datasets_gt"):
        pre_datasets_f1.append(calculate_bertscore_complete(predictions.get("datasets_prediction"), paper.get("datasets_gt"))[2])
    if paper.get("metrics_gt"):
        pre_metrics_f1.append(calculate_bertscore_complete(predictions.get("metrics_prediction"), paper.get("metrics_gt"))[2])

avg_pre_processing_time = sum(pre_processing_times) / len(pre_processing_times)
print(f"{avg_pre_processing_time} seconds")

avg_pre_model_f1 = sum(pre_model_f1) / len(pre_model_f1)
print(f"{avg_pre_model_f1} F1 for model name extraction")

avg_pre_tasks_f1 = sum(pre_tasks_f1) / len(pre_tasks_f1)
print(f"{avg_pre_tasks_f1} F1 for task extraction")

avg_pre_category_f1 = sum(pre_category_f1) / len(pre_category_f1)
print(f"{avg_pre_category_f1} F1 for category extraction")

avg_pre_implementation_f1 = sum(pre_implementation_f1) / len(pre_implementation_f1)
print(f"{avg_pre_implementation_f1} F1 for implementation URL extraction")



avg_pre_dataset_f1 = sum(pre_datasets_f1) / len(pre_datasets_f1)
print(f"{avg_pre_dataset_f1} F1 for dataset extraction")

avg_pre_metric_f1 = sum(pre_metrics_f1) / len(pre_metrics_f1)
print(f"{avg_pre_metric_f1} F1 for metric extraction")




Processing papers with precise pipeline...:   0%|          | 0/156 [00:00<?, ?it/s]

Llama has not returned a legible JSON. It returned: {"
Llama has not returned a legible JSON. It returned: {"
Llama has not returned a legible JSON. It returned: {"
Llama has not returned a legible JSON. It returned: {"
Llama has not returned a legible JSON. It returned: {"
Llama has not returned a legible JSON. It returned: {"
Llama has not returned a legible JSON. It returned: {"
Llama has not returned a legible JSON. It returned: {"
Llama has not returned a legible JSON. It returned: {"
Llama has not returned a legible JSON. It returned: {"
Llama has not returned a legible JSON. It returned: {"
Llama has not returned a legible JSON. It returned: {"
Llama has not returned a legible JSON. It returned: {"key": ["ComplEx", "CP", "TuckER", "RESCAL", "DistMult", "TransE", "Dist-Mult", "RES- CAL", "ComplEx-N3", "CP-N3", "ComplEx", "CP", "TuckER", "RESCAL", "DistMult", "TransE", "Dist-Mult", "RESCAL", "CP", "ComplEx", "TuckER", "DistMult", "TransE", "Dist-Mult", "RESCAL", "CP", "ComplEx", "

In [4]:
eff_model_f1 = []
eff_tasks_f1 = []
eff_category_f1 = []
eff_implementation_f1 = []

eff_datasets_f1 = []
eff_metrics_f1 = []

eff_processing_times = []



for paper in tqdm(clean_dataset, desc="Processing papers with efficient pipeline..."):
    predictions, processing_time = mcg.evaluate_pipeline("efficient", paper)
    eff_processing_times.append(processing_time)
    
    if paper.get("model_name_gt"):
        eff_model_f1.append(calculate_bertscore_complete(predictions.get("model_name_prediction"), paper.get("model_name_gt"))[2])
    if paper.get("tasks_gt"):
        eff_tasks_f1.append(calculate_bertscore_complete(predictions.get("tasks_prediction"), paper.get("tasks_gt"))[2])
    if paper.get("category_gt"):
        eff_category_f1.append(calculate_bertscore_complete(predictions.get("category_prediction"), paper.get("category_gt"))[2])
    if paper.get("implementation_gt"):
        eff_implementation_f1.append(calculate_bertscore_complete(predictions.get("implementation_prediction"), paper.get("implementation_gt"))[2])
    
    
    if paper.get("datasets_gt"): 
        eff_datasets_f1.append(calculate_bertscore_complete(predictions.get("datasets_prediction"), paper.get("datasets_gt"))[2])
    if paper.get("metrics_gt"):
        eff_metrics_f1.append(calculate_bertscore_complete(predictions.get("metrics_prediction"), paper.get("metrics_gt"))[2])


avg_eff_processing_time = sum(eff_processing_times) / len(eff_processing_times)
print(f"{avg_eff_processing_time} seconds")

avg_eff_model_f1 = sum(eff_model_f1) / len(eff_model_f1)
print(f"{avg_eff_model_f1} F1 for model name extraction")

avg_eff_tasks_f1 = sum(eff_tasks_f1) / len(eff_tasks_f1)
print(f"{avg_eff_tasks_f1} F1 for task extraction")

avg_eff_category_f1 = sum(eff_category_f1) / len(eff_category_f1)
print(f"{avg_eff_category_f1} F1 for category extraction")

avg_eff_implementation_f1 = sum(eff_implementation_f1) / len(eff_implementation_f1)
print(f"{avg_eff_implementation_f1} F1 for implementation URL extraction")



avg_eff_dataset_f1 = sum(eff_datasets_f1) / len(eff_datasets_f1)
print(f"{avg_eff_dataset_f1} F1 for dataset extraction")

avg_eff_metric_f1 = sum(eff_metrics_f1) / len(eff_metrics_f1)
print(f"{avg_eff_metric_f1} F1 for metric extraction")

Processing papers with efficient pipeline...:   0%|          | 0/156 [00:00<?, ?it/s]

Llama has not returned a legible JSON. It returned: {"
Llama has not returned a legible JSON. It returned: {"
Llama has not returned a legible JSON. It returned: {"


/home/jovyan/.local/lib/python3.11/site-packages/gliner/data_processing/processor.py:395: UserWarning: Sentence of length 572 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]


Llama has not returned a legible JSON. It returned: {"
Llama has not returned a legible JSON. It returned: {"
Llama has not returned a legible JSON. It returned: {"
Llama has not returned a legible JSON. It returned: {"
Llama has not returned a legible JSON. It returned: {"
Llama has not returned a legible JSON. It returned: {"
Llama has not returned a legible JSON. It returned: {"
12.361485082369585 seconds
0.461165937093588 F1 for model name extraction
0.7905661425529382 F1 for task extraction
0.98457613161632 F1 for category extraction
0.8263349550110953 F1 for implementation URL extraction
0.8308354237904916 F1 for dataset extraction
0.8216213010824643 F1 for metric extraction
